In [ ]:
# ========================================
# E-commerce Analytics Project (PySpark)
# Author: Tomasz Cielepak
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, countDistinct, sum as spark_sum, avg, expr, lit, lag, when, explode, collect_set, to_date
from datetime import timedelta

# ========================================
# 1. Load Data
# ========================================
df = spark.table("workspace.default.ecommerce_sample_2_gb")
df = df.withColumn("event_time", F.to_timestamp("event_time"))

print("\n==== RAW DATA SAMPLE ====")
display(df.limit(5))

# ========================================
# 2. Basic Aggregations by Category
# ========================================
agg_df = (
    df.groupBy("category_code")
      .agg(
          F.count("event_time").alias("num_orders"),
          spark_sum("price").alias("total_amount")
      )
      .orderBy(F.desc("total_amount"))
)

print("\n==== TOTAL ORDERS & REVENUE BY CATEGORY ====")
display(agg_df)

# ========================================
# 3. RFM Analysis
# ========================================
purchases = df.filter(col("event_type") == "purchase")

obs_date = purchases.agg(F.max("event_time")).collect()[0][0] + timedelta(days=1)

rfm = (
    purchases.groupBy("user_id")
             .agg(
                 F.datediff(F.lit(obs_date), F.max("event_time")).alias("Recency"),
                 F.count("event_time").alias("Frequency"),
                 spark_sum("price").alias("Monetary")
             )
)

# Quantiles for scoring
quantiles = rfm.approxQuantile(["Recency", "Frequency", "Monetary"], [0.25, 0.5, 0.75], 0.01)
r_q, f_q, m_q = quantiles

# Helper function
def score(col, q, reverse=False):
    return (
        F.when(col <= q[0], 4 if not reverse else 1)
         .when(col <= q[1], 3 if not reverse else 2)
         .when(col <= q[2], 2 if not reverse else 3)
         .otherwise(1 if not reverse else 4)
    )

rfm_segmented = (
    rfm.withColumn("R_score", score(col("Recency"), r_q, reverse=True))
       .withColumn("F_score", score(col("Frequency"), f_q))
       .withColumn("M_score", score(col("Monetary"), m_q))
       .withColumn("RFM_score", col("R_score")*100 + col("F_score")*10 + col("M_score"))
       .withColumn(
           "segment",
           F.when((col("R_score") >= 3) & (col("F_score") >= 3) & (col("M_score") >= 3), "Champions")
            .when((col("R_score") >= 3) & (col("F_score") >= 2), "Loyal Customers")
            .when((col("R_score") == 4) & (col("F_score") == 1), "New Customers")
            .when((col("R_score") <= 2) & (col("F_score") >= 3), "At Risk")
            .otherwise("Others")
       )
)

print("\n==== RFM SEGMENTS SAMPLE ====")
display(rfm_segmented.limit(10))

summary_rfm = (
    rfm_segmented.groupBy("segment")
    .agg(
        countDistinct("user_id").alias("customers"),
        spark_sum("Monetary").alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

print("\n==== RFM SEGMENT SUMMARY ====")
display(summary_rfm)

# ========================================
# 4. Daily Cohort Retention
# ========================================
purchases = purchases.withColumn("purchase_date", to_date("event_time"))

cohorts = purchases.groupBy("user_id").agg(F.min("purchase_date").alias("cohort_date"))

purchases = purchases.join(cohorts, on="user_id") \
                     .withColumn("days_from_cohort", F.datediff(col("purchase_date"), col("cohort_date")))

retention = (
    purchases.groupBy("cohort_date", "days_from_cohort")
             .agg(countDistinct("user_id").alias("num_users"))
)

cohort_sizes = (
    retention.filter(col("days_from_cohort") == 0)
             .select("cohort_date", col("num_users").alias("cohort_size"))
)

retention = (
    retention.join(cohort_sizes, on="cohort_date")
             .withColumn("retention_rate", col("num_users") / col("cohort_size"))
)

print("\n==== DAILY COHORT RETENTION ====")
display(retention)

# Pivot for matrix view
retention_pivot = (
    retention.groupBy("cohort_date")
             .pivot("days_from_cohort")
             .agg(F.max("retention_rate"))
             .orderBy("cohort_date")
)

print("\n==== RETENTION PIVOT (DAILY) ====")
display(retention_pivot)

# ========================================
# 5. LTV Analysis
# ========================================
ltv_df = purchases.groupBy("user_id").agg(
    spark_sum("price").alias("total_revenue"),
    countDistinct("product_id").alias("purchase_count"),
    F.min("event_time").alias("first_purchase"),
    F.max("event_time").alias("last_purchase")
)

ltv_rfm = ltv_df.join(rfm_segmented, on="user_id", how="left")

ltv_rfm_stats = (
    ltv_rfm.groupBy("segment")
           .agg(
               spark_sum("total_revenue").alias("total_revenue"),
               countDistinct("user_id").alias("num_users"),
               avg("total_revenue").alias("avg_ltv"),
               expr("percentile_approx(total_revenue, 0.5)").alias("median_ltv")
           )
)

print("\n==== LTV BY RFM SEGMENT ====")
display(ltv_rfm_stats)

# ========================================
# 6. Funnel Analysis
# ========================================
steps = ["view", "cart", "purchase"]
funnel = []
for step in steps:
    users = df.filter(col("event_type") == step).agg(countDistinct("user_id")).collect()[0][0]
    funnel.append((step, users))

funnel_df = spark.createDataFrame(funnel, ["step", "num_users"]) \
    .withColumn("step_order", when(col("step") == "view", lit(1))
                             .when(col("step") == "cart", lit(2))
                             .when(col("step") == "purchase", lit(3)))

window = Window.orderBy("step_order")
funnel_df = funnel_df.withColumn("prev_users", lag("num_users").over(window)) \
                     .withColumn("conv_vs_prev", col("num_users") / col("prev_users")) \
                     .withColumn("conv_vs_first", col("num_users") / funnel_df.first()["num_users"])

print("\n==== CONVERSION FUNNEL ====")
display(funnel_df)

# ========================================
# 7. Market Basket Analysis
# ========================================
df_purchases = purchases.select("user_id", "product_id")

user_basket = (
    df_purchases.groupBy("user_id")
                .agg(F.array_sort(collect_set("product_id")).alias("basket"))
                .filter(F.size("basket") >= 1)
)

baskets_expl = user_basket.select("user_id", explode("basket").alias("item"))
pairs = (
    baskets_expl.alias("a")
    .join(baskets_expl.alias("b"), on="user_id")
    .where(col("a.item") < col("b.item"))
    .select(col("a.item").alias("item_1"), col("b.item").alias("item_2"), col("a.user_id"))
)

pair_counts = (
    pairs.groupBy("item_1", "item_2")
         .agg(countDistinct("user_id").alias("users_count"))
         .orderBy(F.desc("users_count"))
         .limit(50)
)

print("\n==== TOP 50 CO-PURCHASED PRODUCT PAIRS ====")
display(pair_counts)

basket_stats = user_basket.agg(
    countDistinct("user_id").alias("num_users_with_basket"),
    F.mean(F.size("basket")).alias("avg_basket_size"),
    F.expr("percentile_approx(size(basket), 0.5)").alias("median_basket_size")
)

print("\n==== BASKET SIZE STATS ====")
display(basket_stats)